# Working with Geodatabases

Most GIS data in ArcGIS Pro is stored in geodatabases. In `arcpy-functions.ipynb`, you listed the files and workspaces in a folder, and `ListWorkspaces()` found the file geodatabases inside it. This notebook looks inside a geodatabase. You will list the feature datasets and feature classes it contains, filter them by name and geometry type, and describe what a feature class holds, so that a script can find and process the data in a geodatabase without typing each name.

🎯 What You'll Learn

* Describe how a geodatabase organizes feature datasets and feature classes
* List the feature datasets in a geodatabase
* List feature classes at the root of a geodatabase and inside a feature dataset
* Filter feature classes by geometry type
* Combine a wildcard, a geometry type, and a feature dataset in one list
* Describe a feature class, including its geometry type and coordinate system
* Running tools on feature class returned in the list

Work through each numbered section: a demo shows how something works, then a ✏️ **Practice** prompt below it gives you a chance to try it yourself.

### ⚙️ Before you start: select the kernel

The **kernel** is the program that runs the Python code in this notebook. Before running any cells, make sure the notebook is using the ArcGIS Pro Python environment. An **environment** is a copy of Python plus the add-on tools installed with it.

Check the kernel name in the top-right corner of the notebook. It should read `arcgispro-py3`, and may include a Python version, for example:
`arcgispro-py3 (Python 3.13.13)`

If it shows a different name or says **Select Kernel**:

1. Click the kernel name (or **Select Kernel**) in the top-right corner
2. Select `arcgispro-py3`
3. If it is not listed, refer back to `jupyter-notebook-basics.ipynb` in `M1-getting-started-with-python\code-alongs`

---

### 1. Setup: ArcPy and the geodatabase workspace

Each notebook runs in its own kernel. Variables, imports, and settings from another notebook do not carry over, so every ArcPy notebook starts the same way that every ArcPy script does: import ArcPy, then set the environment settings.

⚠️ **Important**: If the next cell raises an exception, try these steps:

* Check that the kernel for this notebook is set to `arcgispro-py3`.
* Open ArcGIS Pro and sign in to the `uagis` organization account with your NetID and password so that ArcGIS Pro can assign you a Named User license.

If none of these steps fixes the exception, reach out to your instructor with a screenshot of the error message.

**DO NOT** proceed with this notebook until this exception is resolved.

In [ ]:
# Import the ArcPy site package
import arcpy

In `arcpy-functions.ipynb`, the workspace was a folder. A file geodatabase is also a workspace, so you can set `arcpy.env.workspace` to a geodatabase in the same way.

This notebook explores, and later edits, the City of Austin file geodatabase, `Austin.gdb`, in the `austin_data` folder. Before working with it, we will copy it to the `outputs\m3` folder and set the copy as the workspace. The source data in `austin_data` stays unchanged, and if anything goes wrong in the copy, you can rerun the cells below to replace it with a fresh copy.

⚠️ **Important**: Update the following path to match the location of the working folder you created in `course-data-setup.ipynb`.

In [ ]:
# Import os
import os

# Store the path to your working folder. Update this path to match the
# working folder you created in course-data-setup.ipynb
working_folder = r"C:\workspace\gist"
print(f"working_folder: {working_folder}")

# Build the path to the source geodatabase in the austin_data folder
source_gdb = os.path.join(working_folder, "austin_data", "Austin.gdb")
print(f"source_gdb: {source_gdb}")
print(f"Austin.gdb exists: {arcpy.Exists(source_gdb)}")

# Build the path to the outputs folder for this module, and create it if
# it does not exist yet
output_folder = os.path.join(working_folder, "outputs", "m3")
os.makedirs(output_folder, exist_ok=True)
print(f"output_folder: {output_folder}")

⚠️ **Important**: If `Austin.gdb exists` prints `False`, fix the path in the cell above before continuing.

The **Delete** tool, in the Data Management Tools toolbox, deletes a dataset, including an entire geodatabase. The **Copy** tool, in the same toolbox, copies a dataset to a new location. The next cell deletes any copy left from an earlier run, then makes a fresh copy, so every time you run it you start from the original data.

In [ ]:
# Allow tools to replace outputs that already exist
arcpy.env.overwriteOutput = True
print(f"arcpy.env.overwriteOutput: {arcpy.env.overwriteOutput}")

# Build the path to the copy of the geodatabase in the output folder
austin_gdb = os.path.join(output_folder, "Austin.gdb")
print(f"austin_gdb: {austin_gdb}")

# Delete the copy if it exists from an earlier run
if arcpy.Exists(austin_gdb):
    arcpy.management.Delete(austin_gdb)
    print(f"Deleted the earlier copy: {austin_gdb}")

# Copy the source geodatabase to the output folder
copy_result = arcpy.management.Copy(source_gdb, austin_gdb)
print(f"copy_result[0]: {copy_result[0]}")

# Set the workspace to the copy
arcpy.env.workspace = copy_result[0]
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

⚠️ **Important**: If the Delete tool raises an exception saying the geodatabase is locked, another program is using the copy. Close ArcGIS Pro if it is open, then restart the kernel and rerun the cells in this section from the top.

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 2. What is inside a geodatabase

A **geodatabase** is a workspace that stores GIS data together with the rules for how that data is organized. A **file geodatabase** is stored on disk as a folder whose name ends in `.gdb`. The files inside that folder are managed by ArcGIS, and they do not match up one to one with the datasets you see in ArcGIS Pro. To find the datasets in a geodatabase, ArcPy provides list functions that understand how a geodatabase is organized.

A geodatabase can contain the following:

| Item | What it is |
| --- | --- |
| **Feature dataset** | A container inside a geodatabase that groups thematically related feature classes. All feature classes in a feature dataset share the same coordinate system. |
| **Feature class** | A collection of features that share one geometry type, such as points, lines, or polygons, and the same attribute fields. A feature class is the geodatabase equivalent of a shapefile. |
| **Standalone feature class** | A feature class stored at the **root** of the geodatabase, the top level, rather than inside a feature dataset. |
| **Table** | Attribute data with no geometry. |
| **Raster dataset** | Data stored as a grid of cells, such as an elevation surface or an image. |

A partial view of `Austin.gdb`:

```
Austin.gdb
├── Administrative           (feature dataset)
│   ├── addresspts           (feature class)
│   ├── tracts               (feature class)
│   └── ...
├── Environmental            (feature dataset)
│   ├── watersheds           (feature class)
│   └── ...
├── Parks                    (feature dataset)
├── Transportation           (feature dataset)
├── standalone feature classes
├── tables
└── rasters
```

For more information on geodatabases, refer to [Esri Help Documentation](https://doc.esri.com/en/arcgis-pro/latest/help/data/geodatabases/overview/what-is-a-geodatabase-.html).

This notebook focuses on feature datasets and feature classes. Tables and rasters have list functions of their own.

---

### 3. Listing datasets

`ListWorkspaces()` found the geodatabases inside a folder. One level down, inside a geodatabase, `arcpy.ListDatasets()` returns a list of the names of the datasets in the workspace. A **dataset** is any collection of data that ArcGIS stores and manages as one unit. Feature datasets are one kind of dataset that groups thematically similar feature classes the same coordinate system. A geodatabase can hold other kinds as well, such as raster datasets.

The optional `feature_type` parameter limits the results to one kind of dataset. `feature_type="Feature"` returns only feature datasets. This works the same way as the `workspace_type` parameter of `ListWorkspaces()`: `feature_type` is the second optional parameter, so the clearest way to set it is with a keyword argument.

In [ ]:
# Example 1 - List all datasets in the geodatabase

# We will use the copied Austin.gdb set in arcpy.env.workspace
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

# ListDatasets returns a list object
list_of_datasets = arcpy.ListDatasets()
print(f"type(list_of_datasets): {type(list_of_datasets)}")

# Use a for loop to print each dataset
print(f"Datasets in {arcpy.env.workspace}:")
for i, one_dataset in enumerate(list_of_datasets, start=1):
    print(f"\t{i}. {one_dataset}")

In [ ]:
# Example 2 - List only feature datasets

# ListDatasets returns a list object
list_of_feature_datasets = arcpy.ListDatasets(feature_type="Feature")

# Use a for loop to print each feature dataset
print(f"Feature datasets in {arcpy.env.workspace}:")
for i, one_dataset in enumerate(list_of_feature_datasets, start=1):
    print(f"\t{i}. {one_dataset}")

# Compare this output with the output of Example 1

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 4. Listing feature classes: root vs. feature dataset

`arcpy.ListFeatureClasses()` returns a list of the names of the feature classes in the workspace. Like `ListFiles()` and `ListWorkspaces()`, it searches only the workspace itself. When the workspace is a geodatabase, that means the root of the geodatabase.

Called with no arguments in a geodatabase, `ListFeatureClasses()` returns only the standalone feature classes. The feature classes inside feature datasets are not included.

In [ ]:
# Example 1 - List the standalone feature classes at the root

# We will use variable austin_gdb from Section 1
arcpy.env.workspace = austin_gdb
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

# ListFeatureClasses returns a list object
list_of_root_fcs = arcpy.ListFeatureClasses()
print(f"Number of standalone feature classes: {len(list_of_root_fcs)}")

# Use a for loop to print each standalone feature class
print("Standalone feature classes:")
for i, one_fc in enumerate(list_of_root_fcs, start=1):
    print(f"\t{i}. {one_fc}")

There are two ways to list the feature classes inside a feature dataset:

* Use the optional `feature_dataset` parameter to name the feature dataset. It is the third optional parameter, so set it with a keyword argument: `feature_dataset="Administrative"`.
* Set the workspace to the feature dataset itself. A feature dataset is a workspace inside the geodatabase, and its path is the geodatabase path followed by the feature dataset name.

In [ ]:
# Example 2 - List the feature classes in a feature dataset by name

# ListFeatureClasses returns a list object
list_of_admin_fcs = arcpy.ListFeatureClasses(
    feature_dataset="Administrative"
)
print(f"Number of feature classes: {len(list_of_admin_fcs)}")

# Use a for loop to print each feature class
print("Feature classes in Administrative:")
for i, one_fc in enumerate(list_of_admin_fcs, start=1):
    print(f"\t{i}. {one_fc}")

# Compare this output with the output of Example 1

In [ ]:
# Example 3 - Set the workspace to a feature dataset

# A feature dataset path is the geodatabase path plus the dataset name
parks_dataset = os.path.join(austin_gdb, "Parks")
arcpy.env.workspace = parks_dataset
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

# No feature_dataset argument is needed: the workspace is the dataset
list_of_parks_fcs = arcpy.ListFeatureClasses()

# Use a for loop to print each feature class
print("Feature classes in Parks:")
for i, one_fc in enumerate(list_of_parks_fcs, start=1):
    print(f"\t{i}. {one_fc}")

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 4a] Set the workspace to the Austin geodatabase in the outputs folder, and
# print the workspace.


# 4b] Use arcpy.ListFeatureClasses() with the feature_dataset parameter to
# store a list of the feature classes in the Environmental feature dataset.
# Use a for loop to print the name of each feature class.


💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 5. Filtering by geometry type

In Section 3, the `feature_type` parameter of `ListDatasets()` kept only feature datasets. `ListFeatureClasses()` has a parameter with the same name, but here it filters feature classes by **geometry type**, the kind of shape that every feature in a feature class has. The most common values are:

| `feature_type` value | Returns |
| --- | --- |
| `"Point"` | Point feature classes, such as address points |
| `"Polyline"` | Line feature classes, such as streets |
| `"Polygon"` | Polygon feature classes, such as census tracts |

`feature_type` is the second optional parameter and `feature_dataset` is the third, so set both with keyword arguments. Their order does not matter when you name them.

In [ ]:
# Example 1 - List the polygon feature classes in a feature dataset

# We will use variable austin_gdb from Section 1
arcpy.env.workspace = austin_gdb
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

admin_polygons = arcpy.ListFeatureClasses(
    feature_type="Polygon", feature_dataset="Administrative"
)

# Use a for loop to print each feature class
print("Polygon feature classes in Administrative:")
for i, one_fc in enumerate(admin_polygons, start=1):
    print(f"\t{i}. {one_fc}")

In [ ]:
# Example 2 - List the line feature classes in a feature dataset

transportation_lines = arcpy.ListFeatureClasses(
    feature_type="Polyline", feature_dataset="Transportation"
)

# Use a for loop to print each feature class
print("Line feature classes in Transportation:")
for i, one_fc in enumerate(transportation_lines, start=1):
    print(f"\t{i}. {one_fc}")

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 5a] Use arcpy.ListFeatureClasses() with the feature_type and feature_dataset
# parameters to store a list of all point feature classes in the
# Parks feature dataset. Use a for loop to print each name.


💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 6. Combining a wildcard, a geometry type, and a feature dataset

In `arcpy-functions.ipynb`, you used wildcards with `ListFiles()` to filter file names, such as `"*.shp"` for every name that ends with `.shp`. The first optional parameter of `ListFeatureClasses()` is also a wildcard, `wild_card`, and it filters feature class names with the same `*` pattern. Because `wild_card` is the first parameter, you can pass it without a keyword.

The wildcard, the geometry type, and the feature dataset can be used together. Each filter narrows the list further, so the result contains only the feature classes that match all of them.

In [ ]:
# Example 1 - Filter by name in a feature dataset

# We will use variable austin_gdb from Section 1
arcpy.env.workspace = austin_gdb
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

# Feature classes in Parks with "ball" anywhere in the name
ball_fcs = arcpy.ListFeatureClasses("*ball*", feature_dataset="Parks")

# Use a for loop to print each feature class
print("Feature classes in Parks with 'ball' in the name:")
for i, one_fc in enumerate(ball_fcs, start=1):
    print(f"\t{i}. {one_fc}")

In [ ]:
# Example 2 - Filter by name and geometry type in a feature dataset

# Polygon feature classes in Administrative whose names end in s
s_polygons = arcpy.ListFeatureClasses(
    "*s", feature_type="Polygon", feature_dataset="Administrative"
)

# Use a for loop to print each feature class
print("Polygon feature classes in Administrative whose name ends in s:")
for i, one_fc in enumerate(s_polygons, start=1):
    print(f"\t{i}. {one_fc}")

Alternatively, your code can pass every argument by position and can use an empty string `""` or `None` to skip an optional parameter, such as `arcpy.ListFeatureClasses("", "Polygon", "Administrative")`. It returns the same result.

In [ ]:
# Example 3 - Alternate way to write code

# Using an empty string in place of the wildcard
print("Using empty quotes:")
print(arcpy.ListFeatureClasses("", "Polygon", "Administrative"))

# Using None in place of the wildcard
print("Using None:")
print(arcpy.ListFeatureClasses(None, "Polygon", "Administrative"))

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 6a] Use arcpy.ListFeatureClasses() with a wildcard and the feature_dataset
# parameter to store a list of all the feature classes in Parks whose names 
# contain the word "courts". Use a for loop to print each name.


💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 7. Describing a feature class

The list functions return names. A name tells you what a feature class is called, but not what it contains. `arcpy.Describe()` is an ArcPy function that takes a dataset name or a full path and returns a **Describe object**, an object whose properties describe the dataset.

The properties of a Describe object depend on the kind of data it describes. For a feature class, three useful properties are:

* `dataType`: the kind of dataset, such as `FeatureClass`
* `shapeType`: the geometry type, such as `Point`, `Polyline`, or `Polygon`
* `spatialReference`: the coordinate system of the feature class, as a `SpatialReference` object

`spatialReference` returns an object from the same `SpatialReference` class you used in `arcpy-site-package.ipynb`. There, you created the object from a factory code. Here, ArcPy creates it from the data, and you read its properties, such as `name` and `factoryCode`, in the same way.

In [ ]:
# Example 1 - Describe a feature class

# We will use variable austin_gdb from Section 1
arcpy.env.workspace = austin_gdb
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

# Parcels polygon feature classes in Administrative
parcels_fc_list = arcpy.ListFeatureClasses(
    wild_card="parcels",
    feature_type="Polygon",
    feature_dataset="Administrative"
)

# Extract the one and only item from the list
parcels_fc = parcels_fc_list[0]
print(f"parcels_fc: {parcels_fc}")

fc_desc = arcpy.Describe(parcels_fc)
print(f"type(fc_desc): {type(fc_desc)}")
print(f"Data type: {fc_desc.dataType}")
print(f"Shape type: {fc_desc.shapeType}")

# spatialReference stores a SpatialReference object
fc_sr = fc_desc.spatialReference
print(f"type(fc_sr): {type(fc_sr)}")
print(f"Coordinate system: {fc_sr.name}")
print(f"Factory code: {fc_sr.factoryCode}")

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 7a] Repeat 6a or print the contents of the variable storing the list.


# 7b] Use a for loop on your list from 7a. For each feature class, use
# arcpy.Describe() to print its name, shape type, and coordinate system
# name on one line.


💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 8. Using list results to manage a geodatabase

The list functions in this notebook return names, and a for loop can pass each name to a geoprocessing tool. This is how a script processes, renames, or reorganizes every dataset in a geodatabase without typing each name. Every example in this section changes the copy of `Austin.gdb` from Section 1, so the source data in `austin_data` stays unchanged. To start over, rerun the cells in Section 1.

In [ ]:
# Example 1 - Run a tool on every feature class in a list

# We will use variable austin_gdb from Section 1
arcpy.env.workspace = austin_gdb
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

# We will use variable ball_fcs from Section 6
print("Number of features in each feature class:")
for one_fc in ball_fcs:
    count_result = arcpy.management.GetCount(one_fc)
    feature_count = int(count_result[0])
    print(f"\t{one_fc}: {feature_count}")

The **Rename** tool, in the Data Management Tools toolbox, changes the name of a dataset. The Parks feature dataset includes a feature class named `pavillions`, a misspelling of pavilions.

When a tool changes data inside a feature dataset, use full paths built with `os.path.join()`, so the tool knows exactly which feature dataset holds the data.

In [ ]:
# Example 2 - Rename a feature class

# We will use variable parks_dataset from Section 4
parks_dataset = os.path.join(austin_gdb, "Parks")

old_fc = os.path.join(parks_dataset, "pavillions")
new_fc = os.path.join(parks_dataset, "pavilions")
print(f"old_fc: {old_fc}")
print(f"new_fc: {new_fc}")

# Rename only if the misspelled feature class still exists. Rerunning
# this cell after the rename skips the tool instead of raising an error
if arcpy.Exists(old_fc):
    rename_result = arcpy.management.Rename(old_fc, new_fc)
    print(f"Renamed to: {rename_result[0]}")
else:
    print(f"{old_fc} was not found")

# Confirm the new name
print(arcpy.ListFeatureClasses("pav*", feature_dataset="Parks"))

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this notebook, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

*End of notebook.*